In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

df = pd.read_csv("worldometer_coronavirus_daily_data (1).csv")

# Explore

In [ ]:
print(df.head())
print(df.shape)
print(df.columns)
print(df.dtypes)

# Clean


In [ ]:
print(df.isnull().sum())          # check missing values
df = df.fillna(0)                 # fill missing with 0
df["date"] = pd.to_datetime(df["date"])
df = df.drop_duplicates()

# recoveries = cases - active - deaths
df["recoveries"] = df["cumulative_total_cases"] - df["active_cases"] - df["cumulative_total_deaths"]
df["recoveries"] = df["recoveries"].clip(lower=0)

# Analysis


In [ ]:
# World totals per day
world = df.groupby("date")[["cumulative_total_cases", "recoveries", "cumulative_total_deaths"]].sum()

# Latest data for each country
latest = df.sort_values("date").groupby("country").last()

print("Top 10 countries by cases:")
print(latest["cumulative_total_cases"].nlargest(10))

print("Top 10 countries by deaths:")
print(latest["cumulative_total_deaths"].nlargest(10))

# Compare countries
countries = ["India", "USA", "Brazil", "UK"]
compare = df[df["country"].isin(countries)]

# Visualization (Matplotlib / Seaborn)

In [ ]:
# Cases over time
world["cumulative_total_cases"].plot(title="World Cases Over Time", figsize=(10,4))
plt.show()

# Recoveries over time
world["recoveries"].plot(title="World Recoveries Over Time", color="green", figsize=(10,4))
plt.show()

# Deaths over time
world["cumulative_total_deaths"].plot(title="World Deaths Over Time", color="red", figsize=(10,4))
plt.show()

# Bar chart: country-wise cases
top10 = latest["cumulative_total_cases"].nlargest(10).reset_index()
plt.figure(figsize=(10,4))
sns.barplot(data=top10, x="country", y="cumulative_total_cases")
plt.xticks(rotation=45)
plt.title("Top 10 Countries by Cases")
plt.show()

# Government interventions vs virus spread (India example)
india = df[df["country"] == "India"]
plt.figure(figsize=(10,4))
plt.plot(india["date"], india["daily_new_cases"])
plt.axvline(pd.to_datetime("2020-03-25"), color="red", linestyle="--", label="Lockdown 1 starts")
plt.axvline(pd.to_datetime("2020-06-01"), color="green", linestyle="--", label="Unlock begins")
plt.title("India: Daily Cases vs Government Interventions")
plt.legend()
plt.show()

# Interactive (Plotly)

In [ ]:
# Compare countries over time
fig = px.line(compare, x="date", y="cumulative_total_cases", color="country",
              title="Total Cases: Country Comparison")
fig.show()

# Top 10 deaths
fig = px.bar(latest["cumulative_total_deaths"].nlargest(10).reset_index(),
             x="country", y="cumulative_total_deaths", title="Top 10 Countries by Deaths")
fig.show()


# Thank You !